In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import networks
import glob
import os
import cv2
from torchvision import transforms
from tqdm import tqdm

# 모델 로드
models = {}
models["pose_encoder"] = networks.ResnetEncoder(18, True, 2)
models["pose_encoder"].to("cuda").eval()

models["pose"] = networks.PoseDecoder(models["pose_encoder"].num_ch_enc, num_input_features=1, num_frames_to_predict_for=2)
models["pose"].to("cuda").eval()

pose_encoder_checkpoint = torch.load("/ssd1/jm_data/depth/ssl/depth-hints/logs/jbnu_stereo_teacher_monodepth2/models/weights_24/pose_encoder.pth")
pose_decoder_checkpoint = torch.load("/ssd1/jm_data/depth/ssl/depth-hints/logs/jbnu_stereo_teacher_monodepth2/models/weights_24/pose.pth")
models["pose_encoder"].load_state_dict(pose_encoder_checkpoint)
models["pose"].load_state_dict(pose_decoder_checkpoint)

# 이미지 경로
drive = 13
image_num = 3
# 이미지 경로
path = f"/ssd1/jm_data/depth/ssl/monodepth2/jbnu_stereo/2025_05_08/2025_05_08_drive_{str(drive).zfill(4)}_sync"
image_path = os.path.join(path, f"image_0{image_num}/data/")
image_files = sorted(glob.glob(image_path + "*.png"))  # 정렬 중요

# 저장 경로
save_dir = os.path.join(path, f"extrinsic/image_0{image_num}/data/")
os.makedirs(save_dir, exist_ok=True)

# 이미지 전처리
def load_image(path, height=352, width=640):
    img = cv2.imread(path)
    img = cv2.resize(img, (width, height))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = transforms.ToTensor()(img)
    return img.unsqueeze(0).to("cuda")  # [1, 3, H, W]

# Pose 추정 반복
for i in tqdm(range(len(image_files) - 1)):
    # 연속된 두 프레임 불러오기
    img1 = load_image(image_files[i])
    img2 = load_image(image_files[i + 1])
    # 두 프레임 concat → [1, 6, H, W]
    pose_input = torch.cat([img1, img2], dim=1)
    # PoseNet forward
    feat = [models["pose_encoder"](pose_input)]
    axisangle, translation = models["pose"](feat)

    # extrinsic 변환행렬 (4x4)
    from layers import transformation_from_parameters
    T = transformation_from_parameters(
        axisangle[:, 0], translation[:, 0], invert=False
    )

    # print(f"[{i} → {i+1}] Transformation:\n", T[0].detach().cpu().numpy())
    T_numpy = T[0].detach().cpu().numpy()  # shape: (4, 4)
    save_path = os.path.join(save_dir, f"T_{i:06d}_{i+1:06d}.txt")
    np.savetxt(save_path, T_numpy, fmt="%.6f")
    # print(f"Saved: {save_path}")

 64%|██████▎   | 77/121 [00:01<00:00, 60.04it/s]